# Case Study Thực Hành: Xây Dựng và Huấn Luyện Mini-DLM Bằng PyTorch
### Chuyên khảo: Diffusion Language Models (AIO2026) — AI VIET NAM
**Tác giả:** Phạm Thanh Hưng  
**Mục tiêu:** Cung cấp trải nghiệm thực hành trực quan và khả năng tái lập 100% cho mô hình ngôn ngữ khuếch tán rời rạc (*Discrete Masked Diffusion Language Model*). Sổ tay này chạy mượt mà trên cả **CPU máy cá nhân** (thời gian chạy ~15–25 giây) và **Google Colab với GPU T4** (tự động nhận diện `cuda` và kích hoạt chế độ tăng tốc `fp16/AMP`).

In [1]:
# 1. Kiểm tra và nạp thư viện phụ thuộc (tương thích cả Local CPU và Google Colab)
import sys
import os

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
user_site = os.path.expanduser(f"~/.local/lib/python{py_ver}/site-packages")
if os.path.exists(user_site) and user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    import torch
    import numpy as np
except ImportError:
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip:
            ip.system("pip install -q torch numpy")
        import torch
        import numpy as np
    except Exception:
        pass

print(f"Phiên bản Python: {sys.version.split()[0]}")
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Phiên bản NumPy: {np.__version__}")

Phiên bản Python: 3.12.14
Phiên bản PyTorch: 2.7.0+cpu
Phiên bản NumPy: 2.5.2


## 2. Thiết Lập Thiết Bị và Tính Tái Lập (Reproducibility)
Cố định hạt giống ngẫu nhiên (`seed = 42`) trên toàn bộ các module và tự động phát hiện phần cứng đồ họa.

In [2]:
import math
import time
import random
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Tự động phát hiện GPU hoặc CPU fallback
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

print(f"Thiết bị đang sử dụng: {device}")
print(f"Kích hoạt Automatic Mixed Precision (AMP - fp16): {use_amp}")
if device.type == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

Thiết bị đang sử dụng: cpu
Kích hoạt Automatic Mixed Precision (AMP - fp16): False


## 3. Cấu Hình Kích Thước Mô Hình: Chế Độ QUICK_RUN vs Colab T4
Sổ tay thiết kế sẵn cờ `QUICK_RUN`:
- `QUICK_RUN = True` (mặc định cho CPU cục bộ): Kiến trúc tinh gọn 123.692 tham số, huấn luyện trong ~15 giây trên CPU (phục vụ kiểm thử nhanh và đối chiếu số liệu chính xác trong tài liệu lý thuyết).
- `QUICK_RUN = False` (khuyến nghị cho Colab GPU T4): Mở rộng số kênh `d_model=256`, 8 attention heads, 6 Transformer layers (~1,5–2M tham số), huấn luyện 150 epochs trên dữ liệu phong phú trong ~10–15 phút.

In [3]:
# Cờ cấu hình thực thi
QUICK_RUN = True

if QUICK_RUN:
    print("--- CHẾ ĐỘ: QUICK_RUN (Tối ưu hóa chạy nhanh trên CPU) ---")
    D_MODEL = 64
    NHEAD = 4
    NUM_LAYERS = 3
    DIM_FEEDFORWARD = 128
    BATCH_SIZE = 32
    NUM_EPOCHS = 40
    LEARNING_RATE = 1e-3
    NUM_CORPUS_SAMPLES = 800
    SEQ_LEN = 8
else:
    print("--- CHẾ ĐỘ: FULL COLAB T4 (Mô hình mở rộng với GPU T4) ---")
    D_MODEL = 256
    NHEAD = 8
    NUM_LAYERS = 6
    DIM_FEEDFORWARD = 1024
    BATCH_SIZE = 64
    NUM_EPOCHS = 150
    LEARNING_RATE = 5e-4
    NUM_CORPUS_SAMPLES = 4000
    SEQ_LEN = 8

print(f"Cấu hình: d_model={D_MODEL}, nhead={NHEAD}, layers={NUM_LAYERS}, epochs={NUM_EPOCHS}")

--- CHẾ ĐỘ: QUICK_RUN (Tối ưu hóa chạy nhanh trên CPU) ---
Cấu hình: d_model=64, nhead=4, layers=3, epochs=40


## 4. Xây Dựng Ngữ Vực Sư Phạm Tiếng Việt Có Cấu Trúc Ngữ Pháp
Để người học quan sát rõ nhất tương tác hai chiều (*bidirectional attention*), chúng tôi xây dựng tập dữ liệu cấu trúc câu hoàn chỉnh:
$$\text{[Chủ ngữ]} + \text{"đi"} + \text{[Phương tiện 1]} + \text{[Phương tiện 2]} + \text{"đến"} + \text{[Địa điểm]} + \text{[Hành động]} + \text{[Mục tiêu]}$$
Mối liên kết ràng buộc song phương rất chặt chẽ: `thư_viện` đi liền với `đọc sách`, `công_sở` đi liền với `làm việc`, `bệnh_viện` đi liền với `khám bệnh`. Điều này cho phép kiểm chứng khả năng **điền vào giữa (infilling)** vượt trội của DLM so với mô hình tự hồi quy một chiều.

In [4]:
SUBJECTS = ["nam", "lan", "minh", "an", "hải", "mai", "bình", "hoa"]
PATTERNS = [
    ("xe", "đạp", "trường", "học", "bài"),
    ("xe", "buýt", "thư_viện", "đọc", "sách"),
    ("xe", "máy", "công_sở", "làm", "việc"),
    ("tàu", "hỏa", "đà_nẵng", "du", "lịch"),
    ("máy", "bay", "hà_nội", "công", "tác"),
    ("xe", "buýt", "siêu_thị", "mua", "hàng"),
    ("xe", "đạp", "công_viên", "chụp", "ảnh"),
    ("xe", "máy", "bệnh_viện", "khám", "bệnh"),
]

def build_corpus(num_samples=800):
    sentences = []
    for _ in range(num_samples):
        subj = random.choice(SUBJECTS)
        v1, v2, loc, act, obj = random.choice(PATTERNS)
        sent = [subj, "đi", v1, v2, "đến", loc, act, obj]
        sentences.append(sent)
    return sentences

raw_data = build_corpus(num_samples=NUM_CORPUS_SAMPLES)

# Xây dựng từ vựng độc lập
PAD_TOKEN = "[PAD]"
MASK_TOKEN = "[M]"
UNK_TOKEN = "[UNK]"

vocab = [PAD_TOKEN, MASK_TOKEN, UNK_TOKEN]
for s in raw_data:
    for word in s:
        if word not in vocab:
            vocab.append(word)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
PAD_IDX = word2idx[PAD_TOKEN]
MASK_IDX = word2idx[MASK_TOKEN]

dataset_indices = torch.tensor([[word2idx[w] for w in s] for s in raw_data], dtype=torch.long)

print(f"Tổng số mẫu dữ liệu: {len(raw_data)}")
print(f"Kích thước từ vựng (Vocab Size): {VOCAB_SIZE}")
print(f"Mẫu câu minh họa: {' '.join(raw_data[0])}")

Tổng số mẫu dữ liệu: 800
Kích thước từ vựng (Vocab Size): 44
Mẫu câu minh họa: lan đi xe đạp đến trường học bài


## 5. Kiến Trúc Mạng Nơ-ron: Bidirectional Transformer Encoder & Sinusoidal Time Embedding
Trong DLM, vector biểu diễn mức độ nhiễu $t \in [0, 1]$ được đưa vào mạng thông qua cơ chế mã hóa hình sin (*Sinusoidal Embedding*). Khác với Transformer tự hồi quy sử dụng mặt nạ tam giác dưới (*causal mask*), kiến trúc ở đây là **Transformer Encoder hoàn toàn hai chiều** (*full bidirectional attention*).

In [5]:
class SinusoidalTimeEmbedding(nn.Module):
    """Mã hóa thời gian liên tục t in [0, 1] sang vector d_model."""
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model)
        )

    def forward(self, t):
        half_dim = self.d_model // 2
        emb_scale = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=t.device) * -emb_scale)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return self.mlp(emb)

class MiniMaskedDiffusionLM(nn.Module):
    """Transformer Encoder hai chiều cho Masked Diffusion."""
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=3, dim_feedforward=128, max_len=16):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.time_emb = SinusoidalTimeEmbedding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, t):
        b, l = x.shape
        pos = torch.arange(l, device=x.device).unsqueeze(0).expand(b, l)
        tok_rep = self.token_emb(x) + self.pos_emb(pos)
        t_rep = self.time_emb(t).unsqueeze(1)
        h = tok_rep + t_rep

        out = self.transformer(h)
        logits = self.head(out)
        return logits

## 6. Vòng Lặp Huấn Luyện (Training Procedure)
Tại mỗi bước huấn luyện:
1. Lấy mẫu mức thời gian ngẫu nhiên $t \sim \mathcal{U}(0, 1)$.
2. Quá trình xuôi: Mỗi token bị thay thế bằng $\texttt{[M]}$ với xác suất $t$ (lịch trình tuyến tính).
3. Mạng dự đoán phân phối xác suất và tính hàm mất mát **Cross-Entropy chỉ trên các vị trí thực sự bị che** (công thức rút gọn từ MDLM / MD4).

In [6]:
def train_model():
    model = MiniMaskedDiffusionLM(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        max_len=SEQ_LEN * 2
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Khởi tạo mô hình Mini-DLM thành công với {total_params:,} tham số.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    num_batches = len(dataset_indices) // BATCH_SIZE

    start_time = time.time()
    model.train()

    for epoch in range(1, NUM_EPOCHS + 1):
        perm = torch.randperm(len(dataset_indices))
        epoch_loss = 0.0

        for b in range(num_batches):
            batch_x0 = dataset_indices[perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]].to(device)

            # 1. Lấy mẫu thời gian t ~ U(0, 1)
            t = torch.rand(BATCH_SIZE, device=device)

            # 2. Quá trình xuôi: Mặt nạ hóa ngẫu nhiên với p = t
            mask_prob = t.unsqueeze(1).expand_as(batch_x0)
            is_masked = torch.rand_like(mask_prob) < mask_prob

            # Đảm bảo ít nhất 1 token bị che trên mỗi chuỗi
            no_mask = ~is_masked.any(dim=1)
            if no_mask.any():
                rand_positions = torch.randint(0, SEQ_LEN, (no_mask.sum().item(),), device=device)
                is_masked[no_mask, rand_positions] = True

            batch_xt = batch_x0.clone()
            batch_xt[is_masked] = MASK_IDX

            # 3. Dự đoán và tính loss
            if use_amp:
                with torch.amp.autocast('cuda'):
                    logits = model(batch_xt, t)
                    loss = F.cross_entropy(logits[is_masked], batch_x0[is_masked])
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            else:
                logits = model(batch_xt, t)
                loss = F.cross_entropy(logits[is_masked], batch_x0[is_masked])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / num_batches
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:2d}/{NUM_EPOCHS:2d} | Loss: {avg_loss:.4f} | Elapsed: {time.time()-start_time:.2f}s")

    total_time = time.time() - start_time
    print(f"Huấn luyện hoàn tất trong {total_time:.2f} giây.")
    return model

In [7]:
# Thực thi huấn luyện mô hình
model = train_model()

Khởi tạo mô hình Mini-DLM thành công với 123,692 tham số.


/home/thanhhung27/.cache/uv/archive-v0/5o4ytMeWEidWW7dj/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch  1/40 | Loss: 2.6554 | Elapsed: 0.28s


Epoch 10/40 | Loss: 0.8096 | Elapsed: 2.58s


Epoch 20/40 | Loss: 0.7186 | Elapsed: 4.98s


Epoch 30/40 | Loss: 0.7002 | Elapsed: 7.27s


Epoch 40/40 | Loss: 0.6983 | Elapsed: 10.09s
Huấn luyện hoàn tất trong 10.09 giây.


## 7. Thuật Toán Lấy Mẫu Ngược Với Remasking Dựa Trên Độ Tự Tin
Khác với việc giải mã song song 1 bước dễ gặp lỗi lai ghép (*chimeric failure*), thuật toán giải mã lặp qua $T$ bước khử nhiễu:
- Tại mỗi bước, mạng dự đoán phân phối xác suất cho tất cả các vị trí.
- Sắp xếp các vị trí đang bị che theo độ tin cậy lớn nhất $\max_v p(v)$.
- Gỡ mặt nạ (*unmask*) cho $k$ vị trí có độ tự tin cao nhất và giữ nguyên trạng thái $\texttt{[M]}$ cho các vị trí còn lại.

In [8]:
@torch.no_grad()
def sample_unmasking(model, steps=4, prompt=None):
    model.eval()
    if prompt is None:
        xt = torch.full((1, SEQ_LEN), MASK_IDX, dtype=torch.long, device=device)
        fixed_mask = torch.zeros((1, SEQ_LEN), dtype=torch.bool, device=device)
    else:
        xt = torch.tensor([[word2idx.get(w, MASK_IDX) for w in prompt]], dtype=torch.long, device=device)
        fixed_mask = (xt != MASK_IDX)

    trajectory = []

    current_tokens = [idx2word[idx.item()] for idx in xt[0]]
    trajectory.append({
        "step": 0,
        "t": 1.0,
        "tokens": list(current_tokens),
        "confidences": [1.0 if fixed_mask[0, i] else 0.0 for i in range(SEQ_LEN)],
        "action": "Khởi tạo chuỗi ngẫu nhiên (hoàn toàn [M])"
    })

    for s in range(1, steps + 1):
        t_val = 1.0 - (s - 1) / steps
        t_tensor = torch.tensor([t_val], dtype=torch.float32, device=device)

        logits = model(xt, t_tensor)
        probs = F.softmax(logits[0], dim=-1)
        top_probs, top_preds = probs.max(dim=-1)

        num_to_remain_masked = int(math.floor(SEQ_LEN * (1.0 - s / steps)))
        num_currently_masked = (xt[0] == MASK_IDX).sum().item()
        num_to_unmask = max(0, num_currently_masked - num_to_remain_masked)

        masked_indices = [i for i in range(SEQ_LEN) if xt[0, i] == MASK_IDX and not fixed_mask[0, i]]

        if len(masked_indices) > 0 and num_to_unmask > 0:
            masked_indices.sort(key=lambda idx: top_probs[idx].item(), reverse=True)
            chosen_indices = masked_indices[:num_to_unmask]
            for idx in chosen_indices:
                xt[0, idx] = top_preds[idx]

        current_tokens = [idx2word[idx.item()] for idx in xt[0]]
        conf_list = [round(top_probs[i].item(), 4) for i in range(SEQ_LEN)]

        trajectory.append({
            "step": s,
            "t": round(1.0 - s / steps, 2),
            "tokens": list(current_tokens),
            "confidences": conf_list,
            "action": f"Giải mã {num_to_unmask} token có độ tự tin cao nhất"
        })

    return trajectory

## 8. Thực Nghiệm 1: Quỹ Đạo Khử Mặt Nạ Sinh Không Điều Kiện (T = 4)
Bắt đầu từ chuỗi $x_4 = [\texttt{[M]}, \dots, \texttt{[M]}]$, quan sát từng bước khử nhiễu cho thấy mô hình chốt các từ ngữ khung cấu trúc trước (`đi`, `đến`), sau đó hoàn thiện phương tiện và chủ ngữ/hành động.

In [9]:
print("=" * 70)
print("KỊCH BẢN 1: QUỸ ĐẠO KHỬ MẶT NẠ SINH KHÔNG ĐIỀU KIỆN (T = 4)")
print("=" * 70)
traj1 = sample_unmasking(model, steps=4)
for entry in traj1:
    tok_str = " ".join(f"{w:>9}" for w in entry["tokens"])
    conf_str = " ".join(f"{c:>9.4f}" for c in entry["confidences"])
    print(f"Bước {entry['step']} (t={entry['t']:.2f}):")
    print(f"  Chuỗi token: {tok_str}")
    print(f"  Độ tự tin:   {conf_str}")
    print(f"  Hành động:   {entry['action']}")

KỊCH BẢN 1: QUỸ ĐẠO KHỬ MẶT NẠ SINH KHÔNG ĐIỀU KIỆN (T = 4)
Bước 0 (t=1.00):
  Chuỗi token:       [M]       [M]       [M]       [M]       [M]       [M]       [M]       [M]
  Độ tự tin:      0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    0.0000
  Hành động:   Khởi tạo chuỗi ngẫu nhiên (hoàn toàn [M])
Bước 1 (t=0.75):
  Chuỗi token:       [M]        đi       [M]       [M]       đến       [M]       [M]       [M]
  Độ tự tin:      0.1722    1.0000    0.8360    0.3675    1.0000    0.2493    0.2304    0.2517
  Hành động:   Giải mã 2 token có độ tự tin cao nhất
Bước 2 (t=0.50):
  Chuỗi token:       [M]        đi        xe       máy       đến       [M]       [M]       [M]
  Độ tự tin:      0.1858    0.9999    0.8504    0.3535    1.0000    0.2499    0.2180    0.2692
  Hành động:   Giải mã 2 token có độ tự tin cao nhất
Bước 3 (t=0.25):
  Chuỗi token:       [M]        đi        xe       máy       đến   công_sở       làm       [M]
  Độ tự tin:      0.1828    1.0000    0.9636

## 9. Thực Nghiệm 2: Điền Khuyết Hai Đầu (Infilling / Fill-in-the-Middle)
Đặt bài toán: Ta cố định hai đầu: `lan đi [M] [M] đến thư_viện [M] [M]`.
Nhờ cơ chế chú ý hai chiều toàn cục, mô hình nhìn thấy đồng thời cả tiền tố (`lan đi`) và địa điểm phía sau (`đến thư_viện`), từ đó dự đoán chính xác phương tiện (`xe buýt`) và hành động logic tương ứng (`đọc sách`).

In [10]:
print("=" * 70)
print("KỊCH BẢN 2: ĐIỀN KHUYẾT HAI ĐẦU (INFILLING)")
print("=" * 70)
prompt = ["lan", "đi", "[M]", "[M]", "đến", "thư_viện", "[M]", "[M]"]
print("Prompt đầu vào:", " ".join(prompt))
traj2 = sample_unmasking(model, steps=3, prompt=prompt)
for entry in traj2:
    tok_str = " ".join(f"{w:>9}" for w in entry["tokens"])
    conf_str = " ".join(f"{c:>9.4f}" for c in entry["confidences"])
    print(f"Bước {entry['step']} (t={entry['t']:.2f}):")
    print(f"  Chuỗi token: {tok_str}")
    print(f"  Độ tự tin:   {conf_str}")
    print(f"  Hành động:   {entry['action']}")

KỊCH BẢN 2: ĐIỀN KHUYẾT HAI ĐẦU (INFILLING)
Prompt đầu vào: lan đi [M] [M] đến thư_viện [M] [M]
Bước 0 (t=1.00):
  Chuỗi token:       lan        đi       [M]       [M]       đến  thư_viện       [M]       [M]
  Độ tự tin:      1.0000    1.0000    0.0000    0.0000    1.0000    1.0000    0.0000    0.0000
  Hành động:   Khởi tạo chuỗi ngẫu nhiên (hoàn toàn [M])
Bước 1 (t=0.67):
  Chuỗi token:       lan        đi       [M]       [M]       đến  thư_viện       [M]       [M]
  Độ tự tin:      0.2500    0.9998    0.9928    0.9948    1.0000    0.6024    0.9952    0.9962
  Hành động:   Giải mã 0 token có độ tự tin cao nhất
Bước 2 (t=0.33):
  Chuỗi token:       lan        đi       [M]       [M]       đến  thư_viện       đọc      sách
  Độ tự tin:      0.2488    0.9998    0.9928    0.9949    1.0000    0.5868    0.9954    0.9961
  Hành động:   Giải mã 2 token có độ tự tin cao nhất
Bước 3 (t=0.00):
  Chuỗi token:       lan        đi        xe      buýt       đến  thư_viện       đọc      sách
  Độ tự 

## 10. Phân Tích Kết Quả và Kết Luận
1. **Tính khả thi của Discrete Masked Diffusion:** Mô hình 123k tham số đã hội tụ thành công chỉ sau 40 epoch trên CPU (loss giảm từ $2{,}6554$ xuống $0{,}6983$ trong vòng ~14 giây).
2. **Cơ chế Attention hai chiều:** Mô hình khử nhiễu tự nhiên theo cấu trúc ngữ pháp mà không bị gò bó bởi thứ tự từ trái sang phải như mô hình tự hồi quy.
3. **Hướng dẫn huấn luyện mô hình lớn trên Colab GPU T4:**
   - Chuyển `QUICK_RUN = False` ở Mục 3.
   - Chạy trên Google Colab với Runtime GPU T4 để mô hình huấn luyện với 6 layers, 256 embedding dimensions trên tập văn bản lớn hơn.